# Mini-projet de Probabilités : GCL Randu

### Procédure d'installation

Avant de commencer, il faut installer les modules Python suivant :

* pandas
* matplotlib

Il faut également avoir installé Python3.

### Auteurs

- Menez Ewen, groupe 1A
- Magassouba Ismaela, groupe 1A

### Description

Description

## Ouvrir le fichier

Tout d'abord, on `import` les librairies :

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

Ensuite, on ouvre le fichier csv contenant les données et on stocke celles-ci dans un DataFrame 

In [ ]:
# Lecture du fichier csv et écriture des données dans un DataFrame
df = pd.read_csv("Sunbiz.csv", sep=";", encoding="latin-1")

## Calcul de nouvelles colonnes

Cette étape de préparation de données permet d'enrichir le jeu de données initial en calculant des indicateurs clés nécessaires aux analyses financières et temporelles futures.
* Le montant Total HT combine la valeur brute de la marchandise commandée et ses frais d'expédition associés.
* La rentabilité de livraison crée un indicateur arbitraire pour évaluer le coût unitaire du transport par rapport au volume et prix des produits de la ligne.
* Le Chiffre d'Affaires et la Date isolent la performance commerciale pure et standardisent l'axe temporel pour permettre des groupements chronologiques rigoureux.

In [ ]:
# 1. Calcul du montant Total HT (produits + transport)
df["montant Total HT"] = df["quantité"] * df["Prix unitaire"] + df["Frais envoi"]

# 2. Calcul de la colonne de rentabilité des frais de livraison
df["rentabilite livraison"] = df["Prix unitaire"] * df["quantité"] / df["Frais envoi"]

# 3. Calcul du Chiffre d'Affaires brut (hors frais d'envoi)
df["Chiffre d'Affaires"] = df["Prix unitaire"] * df["quantité"]

# 4. Conversion et standardisation de la date (format JJ/MM/AA)
df["Date commande"] = pd.to_datetime(df["Date commande"], dayfirst=True)

## Calcul des parts du chiffre d'affaires par pays

Pour calculer les parts du chiffre d'affaires total par pays, il faut trouver la part occupée par chaque pays du chiffre d'affaires total.
- On calcule tout d'abord le chiffre d'affaires total en sommant tous les montant total HT
- On sélectionne ensuite tous le chiffre d'affaires réalisé par pays
- On divise enfin ce chiffre d'affaires par le chiffres d'affaires total

Ici, nous avons calculé des parts (pourcentage) du chiffre d'affaire total. Il est alors ici pertinent de représenter ces données sous forme de **diagramme circulaire**.

In [ ]:
montant_total = ( df["montant Total HT"].sum() )
df_pays = ( (df.groupby("PAYS")["montant Total HT"].sum())/ montant_total)

fig, ax = plt.subplots(figsize=(8, 8))

ax.pie(df_pays.values, labels=df_pays.index)

ax.set_title("Part du chiffre d'affaire total par Pays")
plt.show()


## Calcul du chiffre d'affaires total par sous catégorie de produit

Pour calculer le chiffre d'affaires total par sous catégorie de produit, il faut sélectionner tous les chiffres d'affaires réalisés par sous catégorie de produit.
Pour la représentation graphique, un **diagramme à bâtons** est permet de facilement repérer les sous catégories de produit qui produisent un chiffre d'affaire plus ou moins élevé. On assigne également des couleurs allant du vert au violet afin de différencier les sous catégories de produit.

In [ ]:
#constantes de couleur afin de différencier les différentes sous catégories
couleurs = ["#2ecc71", "#35c575", "#3cbfa0", "#44b8b6", "#4bb1cc", "#52aae2", "#5aa3f8", "#619cf2", "#6895e3", "#708ed4", "#7787c5", "#7e80b6", "#8579a7", "#8d7298", "#946b89", "#9b647a", "#9b59b6"]

df_typecli = df.groupby("Sous categorie Produit")["montant Total HT"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(df_typecli.index, df_typecli.values, color=couleurs, edgecolor="black")

ax.set_title("Chiffre d'affaires total par sous catégorie de produit", fontsize=14, fontweight="bold")
ax.set_xlabel("Type de client (Sous-catégorie)", fontsize=12)
ax.set_ylabel("Montant Total HT", fontsize=12)

plt.xticks(rotation=45, ha="right")

plt.tight_layout()  
plt.show()

 ## Évolution du chiffre d'affaires par mois

**Pour calculer et afficher la courbe des ventes on :**
*   Converti les dates des commandes dans un format standard.
*   Regroupe les ventes par période mensuelle.
*   Génère des libellés pour tracer une courbe claire et lisible.

In [ ]:
# 1. Dictionnaire de traduction des mois
mois_fr = {1: "Janvier", 2: "Février", 3: "Mars", 4: "Avril", 5: "Mai", 6: "Juin", 7: "Juillet", 8: "Août", 9: "Septembre", 10: "Octobre", 11: "Novembre", 12: "Décembre"}
# 2. Groupement mensuel, calcul du CA et tri chronologique combiné
df_mensuel = df.groupby(df["Date commande"].dt.to_period("M"))["montant Total HT"].sum().sort_index()
# 3. Génération des labels clairs pour l'axe des abscisses (ex: "Janvier 2025")
labels_propres = [f"{mois_fr[p.month]} {p.year}" for p in df_mensuel.index]
# 4. Initialisation du graphique et tracé de la courbe avec marqueurs
plt.figure(figsize=(12, 6))
plt.plot(labels_propres, df_mensuel.values, marker="o", color="#1f77b4", linewidth=2)
# 5. Habillage et personnalisation (titres, axes, rotation et grille)
plt.title("Évolution du chiffre d'affaires total par mois", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Mois", fontsize=12, labelpad=10)
plt.ylabel("Montant Total HT (€)", fontsize=12, labelpad=10)
plt.xticks(rotation=45)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Distribution des paniers d'achat par pays

Ce graphique en **boîtes à moustaches** (boxplot) permet d'analyser la dispersion et la variabilité des montants des commandes pour chaque marché.
* L'axe des ordonnées liste les pays, classés de manière optimale par ordre croissant de leur valeur médiane.
* L'axe des abscisses indique le montant total du panier d'achat en euros (€).
* Les boîtes à moustaches exposent visuellement la répartition des données (médiane, quartiles et étendue des dépenses) tout en masquant les valeurs aberrantes pour une lecture épurée.

In [ ]:
# 1. Préparation des données (montant des paniers et tri par médiane)
paniers = df.groupby(["PAYS", "Num commande"])["Chiffre d'Affaires"].sum().reset_index()
ord_pays = paniers.groupby("PAYS")["Chiffre d'Affaires"].median().sort_values().index
data = [paniers[paniers["PAYS"] == c]["Chiffre d'Affaires"].values for c in ord_pays]
# 2. Configuration et tracé du boxplot horizontal personnalisé
fig, ax = plt.subplots(figsize=(12, 7))
ax.boxplot(data, vert=False, patch_artist=True, tick_labels=ord_pays, showfliers=False,medianprops={"color": "black", "linewidth": 1.5}, boxprops={"facecolor": "#2a9d8f", "color": "#2a9d8f", "alpha": 0.7})
# 3. Personnalisation des axes et de la grille de lecture arrière
for s in ["top", "right"] :
    ax.spines[s].set_visible(False)
for s in ["left", "bottom"] :
    ax.spines[s].set_color("#cccccc")
ax.xaxis.grid(True, linestyle="--", alpha=0.5, color="#e0e0e0")
# 4. Habillage final (titres, étiquettes et formatage des espaces pour les milliers)
ax.set_title("Distribution des Paniers d'Achat par Pays", fontsize=14, fontweight="bold", pad=20, color="#222222")
ax.set_xlabel("Montant du Panier (€)", fontsize=11, labelpad=10)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{int(x):,}".replace(",", " ")))
plt.tight_layout()
plt.show()

 ## Calcul de la répartition du chiffre d'affaires par catégorie et sous-catégorie

Cette structure de graphiques en **barres horizontales imbriquées** (subplots) permet d'analyser précisément la contribution de chaque sous-catégorie au chiffre d'affaires global.
* La segmentation par sous-graphiques isole proprement chaque grande catégorie de produits pour éviter toute surcharge visuelle.
* L'axe des ordonnées présente de manière claire les sous-catégories, triées par ordre croissant de performance au sein de leur propre groupe.
* L'axe des abscisses et les étiquettes textuelles intégrées affichent directement le montant du chiffre d'affaires en euros (€) pour une lecture à la fois globale et quantitative.

In [ ]:
# 1. Préparation des données et initialisation de la grille multi-graphique
df_g = df.groupby(["Categorie produit", "Sous categorie Produit"])["Chiffre d'Affaires"].sum().reset_index()
cats, cmap = df_g["Categorie produit"].unique(), {"materiel de surf": "#2b7bba", "materiel kite surf": "#e76f51", "materiel planche voile": "#2a9d8f"}
fig, axes = plt.subplots(len(cats), 1, figsize=(12, 10), sharex=True, gridspec_kw={"height_ratios": [len(df_g[df_g["Categorie produit"] == c]) for c in cats]})
max_v = df_g["Chiffre d'Affaires"].max()
# 2. Boucle de tracé par catégorie, style des axes et ajout des étiquettes de texte
for i, cat in enumerate(cats):
    data = df_g[df_g["Categorie produit"] == cat].sort_values(by="Chiffre d'Affaires")
    ax, col = axes[i], cmap.get(cat, "#2b7bba")
    bars = ax.barh(data["Sous categorie Produit"], data["Chiffre d'Affaires"], color=col, height=0.6)
    ax.set_title(f" {cat.upper()}", loc="left", fontsize=12, fontweight="bold", color=col, pad=8)
    for s in ["top", "right"]: ax.spines[s].set_visible(False)
    for s in ["left", "bottom"]: ax.spines[s].set_color("#cccccc")
    ax.xaxis.grid(True, linestyle="--", alpha=0.5, color="#e0e0e0"); ax.set_axisbelow(True)
    for b in bars : 
        ax.text(b.get_width() + (max_v * 0.01), b.get_y() + b.get_height()/2, f"{int(b.get_width()):,}".replace(",", " ") + " €", va="center", fontsize=9.5, fontweight="bold", color="#444444")
# 3. Formatage de l'axe des abscisses commun (X) et affichage final
axes[-1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{int(x):,}".replace(",", " ")))
axes[-1].set_xlabel("Chiffre d'Affaires Total (€)", fontsize=11, labelpad=12, fontweight="bold")
plt.tight_layout()
plt.show()

## Comparaison de la quantité commandée en fonction des frais d'envoi

Tout d'abord, est calculé plus haut la "rentabilité de livraison" d'une ligne de commande.
- Cette dernière permet ici d'associer visuellement les points en assignant cette valeur à leur couleur.
- L'abscisse d'un point correspond aux frais d'envoi de la ligne de commande
- L'ordonnée d'un point correspond à la quantité du produit pour cette ligne de commande

On représente la comparaison entre la quantité commandée et les frais d'envoi avec un **nuage de point** puisque ce graphique permet une visualisation de la répartion des lignes de commandes selon son abscisse et son ordonnée, qui sont ici les frais d'envoi et la quantité commandée.


In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    df["Frais envoi"],
    df["quantité"],

    #on colore le point en fonction de la rentabilité. Les couleurs n'ont pas de sens particulier
    # mais permettent d'associer visuellement les points, en fonction de la similarité d'une commande.
    c=df["rentabilite livraison"],

    #permet un effet de transparence
    alpha=0.2
)

plt.title("Quantité commandée en fonction des frais d'envoi")
plt.xlabel("Frais d'envoi (en €)")
plt.ylabel("Prix unitaire (en €)")

plt.tight_layout()
#plt.show()